# Reading

In [64]:
import pandas as pd
import os

In [65]:
parent_path = "C:/Users/quintyn_come/Codex/AIBT102/"

In [66]:
income_statistics = pd.read_csv(
    os.path.join(
        parent_path,
        "scraping_and_preprocessing/processed_data/BASE_TD_FILO_ESTIMATED_IRIS_2025_DISP_CSV/BASE_TD_FILO_ESTIMATED_IRIS_2025_DISP/DONNEES_IRIS_2025_ESTIMEES.csv"
    ),
    sep=";",
    dtype={"IRIS": str},
    encoding="utf-8-sig"
)

In [67]:
income_statistics.head()

,IRIS,DISP_TP6021,DISP_INCERT21,DISP_Q121,DISP_MED21,DISP_Q321,DISP_EQ21,DISP_D121,DISP_D221,DISP_D321,...,DISP_PPEN21,DISP_PPAT21,DISP_PPSOC21,DISP_PPFAM21,DISP_PPMINI21,DISP_PPLOGT21,DISP_PIMPOT21,DISP_NOTE21,NB_IRIS_2022_SOURCES,COUVERTURE_SURF_2025_PCT
0,010040101,"19,024849247283157","1,9928165359925085","14986,011273811346","20343,6927296171","26137,448272830254","0,5502569193323633","11615,023574122986","14273,574034698138","16074,849072946794",...,"26,89262949441037","6,198951768073184","8,607616122520378","3,3001356550887273","3,8038630421812","1,5036174252504506","-12,50354034551034","0,0",3,"99,9146356541872"
1,010040102,"24,956870148573742","1,0039244938995415","13890,945254082957","18584,657483510193","24775,103571482774","0,5897854235145212","10588,954156970975","12901,698531997845","14672,62262102779",...,"24,906581859750144","5,813549917295208","11,08201002966256","3,696241526719294","5,090751193406806","2,2950173095364605","-12,406036928880667","0,0",3,"96,45392168615366"
2,010040201,"18,425315547991787","1,0057593782760583","15420,42721219933","20935,01783577358","27487,86693381805","0,5777246957911853","11580,542230427549","14284,544039160475","16552,4605597392",...,"27,016156388260864","6,915963846431039","7,513372569150668","2,7510184845523753","3,2097238840330533","1,5526302005652388","-13,975916666019254","0,0",4,"99,99999999999065"
3,010040202,"8,043026264552203","1,0","19582,933721736805","25212,33805870763","33146,712048244626","0,5401544811766336","14796,835919101786","18293,568383088816","20762,68413408753",...,"23,812603695681293","16,16246173966616","4,014703740814159","1,8039708176114395","1,5071775976624504","0,7035553255402698","-17,28639502965582","0,0",3,"99,41187774539077"
4,010070000,"8,0","1,0","19600,0","25230,0","33170,0","0,54","14809,999999999998","18310,0","20780,0",...,"23,800000000000004","16,2","4,0","1,8","1,4999999999999998","0,7","-17,3","0,0",1,"0,010307296993895324"


We want the median income and the poverty rate which corresponds to the columns DISP_TP6021 and DISP_MED21.

In [68]:
income_statistics = income_statistics[["IRIS", "DISP_TP6021", "DISP_MED21"]]

In [69]:
income_statistics = income_statistics.rename(columns={
    'DISP_MED21': 'median_income',
    'DISP_TP6021': 'poverty_rate',
})
for col in ["median_income", "poverty_rate"]:
    income_statistics[col] = pd.to_numeric(
        income_statistics[col]
        .astype(str)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

# Cleaning

Let's check the number of NaN values:

In [70]:
income_statistics.isna().sum()

IRIS               0
poverty_rate     374
median_income    374
dtype: int64

In [71]:
print('Number of rows:', income_statistics.shape[0])

Number of rows: 21058


It seems that for some IRIS we don't have any data, let's drop them.

In [72]:
income_statistics = income_statistics.dropna(
    subset=["IRIS", "median_income", "poverty_rate"]
)

In [73]:
income_statistics.isna().sum()

IRIS             0
poverty_rate     0
median_income    0
dtype: int64

In [74]:
print('Number of rows:', income_statistics.shape[0])

Number of rows: 20684


Just before saving, let's just add an income_statistics_id column

In [75]:
income_statistics.insert(
    0,
    "income_statistics_id",
    range(1, len(income_statistics) + 1)
)

# Saving

In [76]:
output_dir = os.path.join(parent_path, "cleaning/cleaned_data")
output_name = "income_statistics.csv"
output_path = os.path.join(output_dir, output_name)
os.makedirs(output_dir, exist_ok=True)
income_statistics.to_csv(
    output_path,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)